# MiniProject #01: Cobblestone Gifts E-Commerce Data

by Patrick Donnelly & Burke Havranek

EECE 6544: Introduction to Machine Learning and Pattern Recognition

Northeastern University College of Engineering

Summer 2026, Session B

## Scenario

You have just joined the analytics team at **Cobblestone Gifts**, a UK-based online retailer that sells all-occasion
novelty gifts to both wholesale buyers and individual shoppers. The company has never run a formal sales review.
With the financial year closing, the Head of Commercial wants a clear picture of what sold, to whom, and where
but the only thing the IT team can hand you is a single raw export pulled straight from the order system: roughly
half a million transaction lines from December 2010 to December 2011.

The export has never been touched. It mixes completed sales with **cancelled orders and returns**, is missing a
customer ID on a large share of rows, contains **blank product descriptions** and **duplicate lines**, and is peppered
with non-product entries such as postage, bank charges, and manual adjustments. Prices and quantities include
impossible values. In short: **no number in this file can be trusted yet**.

Your job is to clean it. You will profile the data, make and document defensible cleaning decisions, build a tidy
*completed-sales* dataset, and then use it to answer a set of real business questions. The Head of Commercial
will read your findings so every claim must rest on data you can stand behind.

## Part 1: Clean the Data

### Phase A: Load & Profile

#### #3.2: Getting Information

In [77]:
#!/bin/python3
# --Beginning of Code--
import sys, subprocess
def pipq(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", *pkgs])

pipq("scikit-learn", "pandas", "numpy", "matplotlib", "seaborn", "pandas", "kagglehub")

import numpy as np
import pandas as pd
pd.set_option("display.max_columns", 80)
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os

path = kagglehub.dataset_download("carrie1/ecommerce-data");
csv_file = os.path.join(path, "data.csv")
df = pd.read_csv(csv_file, encoding='ISO-8859-1')

In [78]:
print(df.shape) # Note the total rows
N_ROW, _ = df.shape

(541909, 8)


In [79]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [80]:
df.info() # Note the populated rows

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 33.1 MB


In [81]:
df.describe()

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


From the above, we find 541909 total entries in the database, of which six field (InvoiceNo, StockCode, Quantity, InvoiceDate, UnitPrice, Country) are fully-populated with non-null data. Of the remaining two columns, `Description` is missing 1454 entries (~0.268%), whereas `CustomerID` is missing 135080 entries (~24.9%).

#### #3.9 Unique Values

In [82]:
# Naive count of all unique elements in each column
df[["Country", "StockCode", "Description"]].nunique()

Country          38
StockCode      4070
Description    4223
dtype: int64

In [83]:
# Enumeration of each column with and without consideration of null columns
df["Country"].value_counts()

Country
United Kingdom          495478
Germany                   9495
France                    8557
EIRE                      8196
Spain                     2533
Netherlands               2371
Belgium                   2069
Switzerland               2002
Portugal                  1519
Australia                 1259
Norway                    1086
Italy                      803
Channel Islands            758
Finland                    695
Cyprus                     622
Sweden                     462
Unspecified                446
Austria                    401
Denmark                    389
Japan                      358
Poland                     341
Israel                     297
USA                        291
Hong Kong                  288
Singapore                  229
Iceland                    182
Canada                     151
Greece                     146
Malta                      127
United Arab Emirates        68
European Community          61
RSA                         58


In [84]:
df["Country"].value_counts(dropna=False)

Country
United Kingdom          495478
Germany                   9495
France                    8557
EIRE                      8196
Spain                     2533
Netherlands               2371
Belgium                   2069
Switzerland               2002
Portugal                  1519
Australia                 1259
Norway                    1086
Italy                      803
Channel Islands            758
Finland                    695
Cyprus                     622
Sweden                     462
Unspecified                446
Austria                    401
Denmark                    389
Japan                      358
Poland                     341
Israel                     297
USA                        291
Hong Kong                  288
Singapore                  229
Iceland                    182
Canada                     151
Greece                     146
Malta                      127
United Arab Emirates        68
European Community          61
RSA                         58


In [85]:
df["StockCode"].value_counts()

StockCode
85123A    2313
22423     2203
85099B    2159
47566     1727
20725     1639
          ... 
23609        1
85179a       1
23617        1
90214U       1
47591b       1
Name: count, Length: 4070, dtype: int64

In [86]:
df["StockCode"].value_counts(dropna=False)

StockCode
85123A    2313
22423     2203
85099B    2159
47566     1727
20725     1639
          ... 
23609        1
85179a       1
23617        1
90214U       1
47591b       1
Name: count, Length: 4070, dtype: int64

In [87]:
df["Description"].value_counts()

Description
WHITE HANGING HEART T-LIGHT HOLDER    2369
REGENCY CAKESTAND 3 TIER              2200
JUMBO BAG RED RETROSPOT               2159
PARTY BUNTING                         1727
LUNCH BAG RED RETROSPOT               1638
                                      ... 
LETTER "U" BLING KEY RING                1
wet                                      1
wet boxes                                1
????damages????                          1
lost                                     1
Name: count, Length: 4223, dtype: int64

In [88]:
df["Description"].value_counts(dropna=False)

Description
WHITE HANGING HEART T-LIGHT HOLDER    2369
REGENCY CAKESTAND 3 TIER              2200
JUMBO BAG RED RETROSPOT               2159
PARTY BUNTING                         1727
LUNCH BAG RED RETROSPOT               1638
                                      ... 
LETTER "U" BLING KEY RING                1
wet                                      1
wet boxes                                1
????damages????                          1
lost                                     1
Name: count, Length: 4224, dtype: int64

In [89]:
# Isolate non product codes
non_product = df[df["StockCode"].isin(["POST", "DOT", "M", "BANK CHARGES", "AMAZONFEE", "D"])]


In [90]:
non_product["StockCode"].value_counts()

StockCode
POST            1256
DOT              710
M                571
D                 77
BANK CHARGES      37
AMAZONFEE         34
Name: count, dtype: int64

#### #3.8: min/max/sum/mean/count

In [91]:
# Summarize quantity and unit price data (including returns)
df[["Quantity", "UnitPrice"]].describe()

,Quantity,UnitPrice
count,541909.000000,541909.000000
mean,9.552250,4.611114
std,218.081158,96.759853
min,-80995.000000,-11062.060000
25%,1.000000,1.250000
50%,3.000000,2.080000
75%,10.000000,4.130000
max,80995.000000,38970.000000


In [92]:
# Identify returns
returns = df[df["Quantity"] < 0]
returns

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,12/1/2010 9:41,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,12/1/2010 9:49,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,12/1/2010 10:24,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,12/1/2010 10:24,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,12/1/2010 10:24,0.29,17548.0,United Kingdom
...,...,...,...,...,...,...,...,...
540449,C581490,23144,ZINC T-LIGHT HOLDER STARS SMALL,-11,12/9/2011 9:57,0.83,14397.0,United Kingdom
541541,C581499,M,Manual,-1,12/9/2011 10:28,224.69,15498.0,United Kingdom
541715,C581568,21258,VICTORIAN SEWING BOX LARGE,-5,12/9/2011 11:57,10.95,15311.0,United Kingdom
541716,C581569,84978,HANGING HEART JAR T-LIGHT HOLDER,-1,12/9/2011 11:58,1.25,17315.0,United Kingdom


In [74]:
# Identify invalid prices
invalid_prices = df[df["UnitPrice"] < 0]
invalid_prices

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299983,A563186,B,Adjust bad debt,1,8/12/2011 14:51,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,8/12/2011 14:52,-11062.06,NaN,United Kingdom


#### # 3.1 Creating a DataFrame

In [ ]:
# Dictionary for right now to consolidate some regions
country_to_region = {
    # UK/Ireland
    "United Kingdom": "UK&IE",
    "EIRE": "UK&IE",
    "Channel Islands": "UK&IE",

    # Mainland Europe (core)
    "Germany": "Continental Europe",
    "France": "Continental Europe",
    "Spain": "Continental Europe",
    "Netherlands": "Continental Europe",
    "Belgium": "Continental Europe",
    "Switzerland": "Continental Europe",
    "Portugal": "Continental Europe",
    "Italy": "Continental Europe",
    "Austria": "Continental Europe",
    "Denmark": "Continental Europe",
    "Norway": "Continental Europe",
    "Sweden": "Continental Europe",
    "Finland": "Continental Europe",
    "Iceland": "Continental Europe",
    "Cyprus": "Continental Europe",
    "Greece": "Continental Europe",
    "Malta": "Continental Europe",
    "Lithuania": "Continental Europe",
    "Poland": "Continental Europe",
    "Czech Republic": "Continental Europe",
    "European Community": "Continental Europe",

    # Middle East
    "Israel": "Middle East",
    "Lebanon": "Middle East",
    "United Arab Emirates": "Middle East",
    "Bahrain": "Middle East",
    "Saudi Arabia": "Middle East",

    # Africa
    "RSA": "Africa",

    # Americas
    "USA": "Americas",
    "Canada": "Americas",
    "Brazil": "Americas",

    # Oceania
    "Australia": "Oceania",
    "Japan": "Oceania",
    "Hong Kong": "Oceania",
    "Singapore": "Oceania",

    # Unknown
    "Unspecified": "Unspecified"
}

### Phase B: Select & filter

#### #3.3: Slicing

#### #3.4: Conditional Selection

#### #3.5: Sorting Values

### Phase C: Clean & Fix

#### #3.6: Replacing Values

#### #3.7: Renaming Columns

#### #3.10: Handling Missing Values

#### #3.11: Deleting a Column

#### #3.12: Deleting a Row

### Phase D: Engineer & Summarize

#### #3.18: Applying a Function

#### #3.17: Looping over a Column

#### #3.14: Grouping by Values

#### #3.16: Aggregating

#### #3.19: Applying to Groups

#### #3.15: Grouping by Time

### Phase E: Combine

#### #3.20: Concatenating

#### #3.21: Merging

## Part 2: Business Questions

### 1. Seasonality

### 2. Best Sellers

### 3. Markets

### 4. Customer Concentration

### 5. Order Value

### 6. Returns & Cancellations

### 7. Data-Quality Memo